Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)
A) 30 ngày
B) 90 ngày
C) 144 ngày
D) 365 ngày

In [30]:
import pandas as pd
df=pd.read_csv('orders.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values(by=['customer_id', 'order_date'])
df['inter_order_gap'] = df.groupby('customer_id')['order_date'].diff().dt.days
valid_gaps = df['inter_order_gap'].dropna()
median_gap = valid_gaps.median()
print(f"Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) là: {median_gap} ngày")

Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) là: 144.0 ngày


Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?
A) Premium
B) Performance
C) Activewear
D) Standard

In [2]:
import pandas as pd
df_products = pd.read_csv('products.csv')
df_products.head()
df_products['gross_margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']
segment_margin = df_products.groupby('segment')['gross_margin'].mean()
segment_margin_sorted = segment_margin.sort_values(ascending=False)
print(segment_margin_sorted)

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64


Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?
A) defective
B) wrong_size
C) changed_mind
D) not_as_described

In [3]:
import pandas as pd
df_returns = pd.read_csv('returns.csv')
df_products = pd.read_csv('products.csv')
result = pd.merge(df_returns, df_products, left_on="product_id", right_on="product_id", how="inner")
result = result.groupby(result['segment']=='Streetwear')['return_reason'].value_counts()
print(result)


segment  return_reason   
False    wrong_size          13967
         defective            8020
         not_as_described     7035
         changed_mind         6931
         late_delivery        3986
Name: count, dtype: int64


Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?
A) organic_search
B) paid_search
C) email_campaign
D) social_media

In [4]:
import pandas as pd
df_webtraffic = pd.read_csv('web_traffic.csv')
df_webtraffic.head()
result=df_webtraffic.groupby('traffic_source')['bounce_rate'].mean().sort_values(ascending=True)
print(result)

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64


Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?
A) 12%
B) 25%
C) 39%
D) 54%

In [5]:
import pandas as pd
df_orderitems = pd.read_csv('order_items.csv')
all=len(df_orderitems)
promo_applied = df_orderitems['promo_id'].notnull().sum()
percentage = (promo_applied / all) * 100
print(f"{percentage:.0f}%")

39%


C:\Users\USER\AppData\Local\Temp\ipykernel_12744\141238267.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_orderitems = pd.read_csv('order_items.csv')


Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)
A) 55+
B) 25–34
C) 35–44
D) 45–54

In [6]:
import pandas as pd
df_customers = pd.read_csv('customers.csv')
df_orders=pd.read_csv('orders.csv')
result = pd.merge(df_orders, df_customers, on='customer_id')
total_orders_by_age_group = result['age_group'].value_counts()
total_customers_by_age_group = df_customers.groupby('age_group')['customer_id'].count()
avg_orders = total_orders_by_age_group / total_customers_by_age_group
print(avg_orders.sort_values(ascending=False))

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64


Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?
A) West
B) Central
C) East
D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

Phân chia dữ liệu cho bài toán dự báo:
• sales_train.csv: 04/07/2012 → 31/12/2022
• sales_test.csv: 01/01/2023 → 01/07/2024

In [ ]:
import pandas as pd
df_geography = pd.read_csv('geography.csv')
df_orders = pd.read_csv('orders.csv')
df_payments = pd.read_csv('payments.csv')
result1 = pd.merge(df_orders, df_payments, on='order_id')
result2 = pd.merge(df_orders, df_geography, on='zip')
result = pd.merge(result1, result2, on='order_id')

result

sales_train=result[(result['order_date_x'] >= '2012-07-04') & (result['order_date_x'] <= '2022-12-31')]
sales_train.to_csv('sales_train.csv', index=False)
print('Dataset saved as sales_train.csv')

sales_test=result[(result['order_date_x'] >= '2023-01-01') & (result['order_date_x'] <= '2024-07-01')]
sales_test.to_csv('sales_test.csv', index=False)
print('Dataset saved as sales_test.csv')


Dataset saved as sales_train.csv
Dataset saved as sales_test.csv


In [8]:
import pandas as pd
df_sales_train = pd.read_csv('sales_train.csv')
df_sales_train.head()
df_sales_train.groupby('region')['payment_value'].sum().sort_values(ascending=False)


region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64

Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?
A) credit_card
B) cod
C) paypal
D) bank_transfer

In [9]:
import pandas as pd
df_orders = pd.read_csv('orders.csv')
cancelled_orders = df_orders[df_orders['order_status'] == 'cancelled']
payment_methods = cancelled_orders['payment_method'].value_counts()
print(payment_methods)

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?
A) S
B) M
C) L
D) XL

In [23]:
import pandas as pd
df_order_items = pd.read_csv('order_items.csv')
df_products= pd.read_csv('products.csv')
df_returns = pd.read_csv('returns.csv')

df_all_sales = pd.merge(df_order_items, df_products[['product_id', 'size']], on='product_id')
total_sold = df_all_sales.groupby('size').size()
df_check_return = pd.merge(df_all_sales, df_returns[['order_id']], on='order_id', how='inner')
total_returned = df_check_return.groupby('size').size()
return_rate = (total_returned / total_sold).sort_values(ascending=False)
print(return_rate)

C:\Users\USER\AppData\Local\Temp\ipykernel_12744\3403959738.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_order_items = pd.read_csv('order_items.csv')


size
L     0.069549
M     0.067433
S     0.065943
XL    0.065100
dtype: float64


Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất?
A) 1 kỳ (trả một lần)
B) 3 kỳ
C) 6 kỳ
D) 12 kỳ

In [28]:
import pandas as pd
df_payments=pd.read_csv('payments.csv')
df_payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64